# AutoCytotox Simple Example

This notebook runs the same YAML-driven workflow as `autocytotox.py` on the bundled `example_data/` folder. It is intended as a first end-to-end check before you point the config at your own FCS files.

## Before You Run

Create the project environment once from the repository root, then select the `autogating_env` kernel for this notebook:

```bash
conda env create -f environment.yml
conda activate autogating_env
```

`environment.yml` is the canonical environment definition for this project.

In [21]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / 'autocytotox.py').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config_loader import apply_runner_overrides, load_runner_config
from src.gridsearch import expand_gridsearch_configurations, load_gridsearch_config
from src.identifiers import preview_identifier_groups, resolve_identifier_handler
from src.workflow.runner import run_from_config

config_path = repo_root / 'examples' / 'simple_example_config.yaml'
example_data = repo_root / 'example_data'


config_path, example_data

(WindowsPath('c:/Users/aszarzynski/Github/AutoCytotox/examples/simple_example_config.yaml'),
 WindowsPath('c:/Users/aszarzynski/Github/AutoCytotox/example_data'))

## Inspect the Example Data

AutoCytotox discovers folders whose names start with `Day `. Each folder contains matched effector-only, target-only, and coculture FCS files.

In [22]:
day_folders = sorted(p for p in example_data.iterdir() if p.is_dir() and p.name.startswith('Day '))

rows = []
for folder in day_folders:
    names = [p.name for p in folder.glob('*.fcs')]
    rows.append({
        'folder': folder.name,
        'total_fcs': len(names),
        'coculture_files': sum('CC_' in name for name in names),
        'effector_only_files': sum('NK' in name for name in names),
        'target_only_files': sum('K562' in name for name in names),
        'beads_files': sum('Beads' in name for name in names),
    })

pd.DataFrame(rows)

,folder,total_fcs,coculture_files,effector_only_files,target_only_files,beads_files
0,Day 0_B,31,18,6,6,1
1,Day 1_B,31,18,6,6,1
2,Day 2_R,31,18,6,6,1
3,Day 6_R,31,18,6,6,1
4,Day 7_R,31,18,6,6,1


## Load the YAML Configuration

`examples/simple_example_config.yaml` contains the validated best-workflow defaults. You can copy it for your own data and edit `paths.data_path`, `paths.output_path`, `channels.*`, and the filename identifier strategy if your files use a different grouping pattern.

In [23]:
config_data = load_runner_config(str(config_path))

pd.DataFrame([
    {'section': 'workflow', 'parameter': 'method', 'value': config_data['workflow']['method']},
    {'section': 'gating', 'parameter': 'percent', 'value': config_data['gating']['percent']},
    {'section': 'gating', 'parameter': 'bw_adjust', 'value': config_data['gating']['bw_adjust']},
    {'section': 'gating', 'parameter': 'min_distance', 'value': config_data['gating']['min_distance']},
    {'section': 'gating', 'parameter': 'gate2_method', 'value': config_data['gating']['gate2_method']},
    {'section': 'gating', 'parameter': 'gate_derivation', 'value': config_data['gating']['gate_derivation']},
    {'section': 'gating', 'parameter': 'percentile_q', 'value': config_data['gating']['percentile_q']},
    {'section': 'gating', 'parameter': 'beads_x_cutoff', 'value': config_data['gating']['beads_x_cutoff']},
])

,section,parameter,value
0,workflow,method,euclidean
1,gating,percent,0.95
2,gating,bw_adjust,0.05
3,gating,min_distance,30
4,gating,gate2_method,ratio
5,gating,gate_derivation,percentile
6,gating,percentile_q,0.9
7,gating,beads_x_cutoff,2400000


## Preview Filename Grouping

The default identifier strategy is `well_row`. It maps the trailing well code in each filename to a row letter so matched files are analyzed together.

In [24]:
demo_folder = day_folders[0]
demo_files = sorted(p.name for p in demo_folder.glob('*.fcs') if 'Beads' not in p.name)

handler = resolve_identifier_handler(
    strategy=config_data['workflow']['identifier_strategy'],
    callable_path=config_data['workflow']['identifier_callable'],
    repo_root=str(repo_root),
)
groups = preview_identifier_groups(demo_files, handler)

pd.DataFrame(
    [{'identifier': key, 'n_files': len(value), 'files': ', '.join(value[:4])} for key, value in groups.items()]
)

,identifier,n_files,files
0,D,5,"03-CC_B_Opt_01-D1.fcs, 03-CC_B_Opt_01-D2.fcs, ..."
1,E,5,"03-CC_B_Opt_02-E1.fcs, 03-CC_B_Opt_02-E2.fcs, ..."
2,F,5,"03-CC_B_Opt_02-F1.fcs, 03-CC_B_Opt_02-F2.fcs, ..."
3,A,5,"03-CC_B_Ref_02-A1.fcs, 03-CC_B_Ref_02-A2.fcs, ..."
4,B,5,"03-CC_B_Ref_02-B1.fcs, 03-CC_B_Ref_02-B2.fcs, ..."
5,C,5,"03-CC_B_Ref_03-C1.fcs, 03-CC_B_Ref_03-C2.fcs, ..."


## Custom Identifier Callable

`identifier_strategy` selects one of the built-in grouping rules. `well_row` is the default because the bundled filenames end with a plate-well token such as `E10(6)`, and the row letter `E` is the experimental group.

`identifier_callable` is for filename layouts that do not match the built-ins. It points to a Python function using `module:function` or `path/to/file.py:function`. The function must accept a list of filenames and return one group label per filename. When `identifier_callable` is set, it overrides `identifier_strategy`.

In [25]:
def custom_identifier_by_trailing_row(filename_list: list[str]) -> list[str]:
    """Example custom callable with the signature AutoCytotox expects."""
    labels = []
    for filename in filename_list:
        stem = Path(filename).stem
        trailing_token = stem.split('-')[-1]  # e.g. E10(6), AX, or A4
        labels.append(trailing_token[0])      # e.g. E, A, or A
    return labels

custom_groups = preview_identifier_groups(demo_files, custom_identifier_by_trailing_row)

pd.DataFrame([
    {
        'purpose': 'built-in default',
        'yaml': 'identifier_strategy: well_row',
        'example_group_count': len(groups),
    },
    {
        'purpose': 'custom callable file',
        'yaml': 'identifier_callable: examples/custom_identifier.py:custom_identifier_by_trailing_row',
        'example_group_count': len(custom_groups),
    },
])

,purpose,yaml,example_group_count
0,built-in default,identifier_strategy: well_row,6
1,custom callable file,identifier_callable: examples/custom_identifie...,6


## Run the Workflow

This cell runs the full example using the plotting settings from `examples/simple_example_config.yaml`. With `plotting.enable_all: true`, every diagnostic plot is generated under `paths.output_path`.

In [26]:
runtime_config = apply_runner_overrides(
    config_data,
    quiet=True,
)

results = run_from_config(runtime_config, config_path=str(config_path))
raw_results = results['combined_results']
aggregated = results['auto_agg']

raw_results.shape, aggregated.shape

((150, 17), (10, 11))

## Review Results

`combined_results` contains per-file quadrant and cytotoxicity values. `auto_agg` contains day, condition, Opt/Ref aggregates with mean, standard deviation, and CV percent.

In [27]:
cytotox_table = (
    raw_results.loc[raw_results['Cytotoxicity'].notna(), ['Folder', 'File', 'Cytotoxicity']]
    .sort_values(['Folder', 'File'])
    .reset_index(drop=True)
)

cytotox_table.head(12)

,Folder,File,Cytotoxicity
0,Day 0_B,03-CC_B_Opt_01-D1.fcs,0.000000
1,Day 0_B,03-CC_B_Opt_01-D2.fcs,0.000000
2,Day 0_B,03-CC_B_Opt_01-D3.fcs,0.000000
3,Day 0_B,03-CC_B_Opt_02-E1.fcs,0.000000
4,Day 0_B,03-CC_B_Opt_02-E2.fcs,0.000000
5,Day 0_B,03-CC_B_Opt_02-E3.fcs,0.000000
6,Day 0_B,03-CC_B_Opt_02-F1.fcs,0.000000
7,Day 0_B,03-CC_B_Opt_02-F2.fcs,0.000000
8,Day 0_B,03-CC_B_Opt_02-F3.fcs,0.000000
9,Day 0_B,03-CC_B_Ref_02-A1.fcs,25.243402


In [28]:
aggregated[['folder', 'condition', 'opt_ref', 'n_values', 'cytotoxicity_mean', 'cytotoxicity_std', 'cv_percent']]

,folder,condition,opt_ref,n_values,cytotoxicity_mean,cytotoxicity_std,cv_percent
0,Day 0_B,B,Opt,9,0.000000,0.000000,NaN
1,Day 0_B,B,Ref,9,26.929211,3.265350,12.125679
2,Day 1_B,B,Opt,9,22.087876,3.392954,15.361159
3,Day 1_B,B,Ref,9,49.056048,5.927740,12.083606
4,Day 2_R,R,Opt,9,57.652291,1.163997,2.018995
5,Day 2_R,R,Ref,9,63.918617,1.517409,2.373971
6,Day 6_R,R,Opt,9,6.946246,19.414867,279.501576
7,Day 6_R,R,Ref,9,45.541476,8.680235,19.060065
8,Day 7_R,R,Opt,9,17.756502,6.173636,34.768309
9,Day 7_R,R,Ref,9,42.278164,32.483046,76.831734


## Output Workbook

The workflow writes a timestamped Excel workbook and, when enabled, plot folders under `paths.output_path`.

In [29]:
excel_path = Path(results['excel_path'])
print(f'Excel results: {excel_path}')
print(f'Output folder: {excel_path.parent}')

pd.ExcelFile(excel_path).sheet_names

Excel results: C:\Users\aszarzynski\Github\AutoCytotox\output\examples\autocytotox_results_euclidean_95_28042026_225921.xlsx
Output folder: C:\Users\aszarzynski\Github\AutoCytotox\output\examples


['Raw_Results', 'Aggregated', 'Parameters']

## Preview the Smoke Grid

The grid-search config in this repository is a one-configuration smoke grid matching the validated best workflow. This cell only expands and previews the grid; it does not run the grid search.

In [30]:
grid_config_path = repo_root / 'gridsearch_config.yaml'
grid_config = load_gridsearch_config(str(grid_config_path))
grid_entries = expand_gridsearch_configurations(grid_config)

pd.DataFrame(grid_entries)

,method,percent,bw_adjust,min_distance,gate2_method,gate_derivation,percentile_q,k_sigma
0,euclidean,0.95,0.05,30,ratio,percentile,0.9,2.5


## Equivalent Commands

```bash
python autocytotox.py --config examples/simple_example_config.yaml --quiet
python autocytotox_gridsearch.py --config gridsearch_config.yaml --dry-run
```

For your own data, copy the example YAML file, edit the paths and channel labels, then run the first command with your config path.